# covattn - run experiments on Colab
Runtime -> Change runtime type -> **A100 GPU**. Run cells 1-5 once per session; every experiment cell uses `--resume`, so after a disconnect just re-run 1-5 and the experiment cell.

Scripts run as subprocesses (`!python ...`), so pinned packages take effect without restarting the runtime.

In [ ]:
# 1. mount Drive (results live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. get / update the code from GitHub
REPO = 'https://github.com/FlyMe2star/covattn.git'
BRANCH = 'main'
import os
if not os.path.isdir('/content/covattn/.git'):
    !git clone -q -b {BRANCH} {REPO} /content/covattn
else:
    !git -C /content/covattn fetch -q origin && git -C /content/covattn reset -q --hard origin/{BRANCH}
%cd /content/covattn
!git log --oneline -3

In [ ]:
# 3. install pinned dependencies (keeps Colab's own CUDA torch)
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
# Colab ships torchao 0.10, which peft 0.21 rejects when it builds LoRA layers; nothing here needs torchao
!pip uninstall -y -q torchao

In [ ]:
# 4. GPU check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch, chronos; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-', '| chronos', chronos.__version__)"

In [ ]:
# 5. results directory on Drive
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/pakdd27/results'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.makedirs(os.environ['RESULTS_DIR'], exist_ok=True)
print(os.environ['RESULTS_DIR'])

## Smoke test (about 2 min): unit + model tests, source checks, Exp1 quick

In [ ]:
!python -m pytest -q
!python scripts/verify_model.py > /dev/null && echo verify_model OK
!python scripts/exp1_phenomenon.py --quick --resume --run-name colab_quick 2>&1 | grep -E 'ESTIMATED|done|Error|Traceback'

## Stage 0 - Exp1 (phenomenon)
The canonical run `exp1/main` (304 cells) is already on Drive; this cell only checks it.

In [ ]:
!ls $RESULTS_DIR/exp1/main && echo "cells: $(ls $RESULTS_DIR/exp1/main/cells | wc -l)"
!python scripts/exp1_phenomenon.py --summary-only && head -30 $RESULTS_DIR/reports/exp1_summary_main.md

In [ ]:
# complete / re-run Exp1 on this GPU (resumable; skips finished cells)
# !python scripts/exp1_phenomenon.py --resume --batch-size 256

## Stage 1 - Exp2 (mechanism: decomposition, patching, REG sink, cross-gain preview)
Val split. 48 cells = 6 systems x L{512,2048} x {prbs, multisine} x {step, pulse}, 64 pairs each. The script prints its own time estimate for this GPU (expect roughly 1 h on L4).

In [ ]:
!python -m pytest -q tests/test_hooks.py
!python scripts/exp2_mechanism.py --quick --resume --run-name colab_quick 2>&1 | grep -E 'ESTIMATED|done|Error|Traceback'

In [ ]:
!python scripts/exp2_mechanism.py --resume 2>&1 | grep -v -E 'Loading weights|UserWarning|warnings.warn'

In [ ]:
!head -120 $RESULTS_DIR/reports/exp2_summary_main.md

## Stage 2 - Exp3 (repairs vs baselines, synthetic test) and Exp4 (real covariate + univariate)
Exp3 runs: R1 selection on val -> dither selection -> R2 gate learning -> full FT + LoRA (official `fit`) -> test evaluation (152 cells). Each stage is resumable. Exp4 reuses the Exp3 artefacts.

In [ ]:
!python -m pytest -q tests/test_repair.py
!python scripts/exp3_repair.py --quick --resume 2>&1 | grep -E 'ESTIMATED|chosen|done|Error|Traceback'
!python scripts/exp4_real_and_univariate.py --quick --resume 2>&1 | grep -E 'done|Error|Traceback'

Run `v2` (current): R2 with 32 pairs/step, lr 0.01, 400 steps, pull towards 1; fine-tuning in 64-row micro-batches x 4 (the first run `main` stopped at the start of full FT, suspected OOM). A fine-tuning failure is now logged to log.txt and the rest continues.

In [ ]:
!python scripts/exp3_repair.py --resume --run-name v2 2>&1 | grep -v -E 'Loading weights|Writing model|UserWarning|FutureWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed|Traceback' $RESULTS_DIR/exp3/v2/log.txt; ls $RESULTS_DIR/exp3/v2

In [ ]:
!python scripts/exp4_real_and_univariate.py --resume --run-name v2 --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'

In [ ]:
!head -80 $RESULTS_DIR/reports/exp3_summary_v2.md; cat $RESULTS_DIR/reports/exp4_summary_v2.md

LoRA for run `v2` (it failed there on Colab's torchao 0.10; fixed in cell 3). This trains LoRA only (full FT and the other methods are reused), adds it to the 152 finished cells with the same samples, then re-runs Exp4 as `v2lora` (Exp4 is only a few minutes).

In [ ]:
!python scripts/exp3_repair.py --resume --run-name v2 --add-methods ft_lora 2>&1 | grep -v -E 'Loading weights|Writing model|UserWarning|FutureWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed|Traceback|added' $RESULTS_DIR/exp3/v2/log.txt | tail -5

In [ ]:
!python scripts/exp4_real_and_univariate.py --resume --run-name v2lora --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'
!head -40 $RESULTS_DIR/reports/exp3_summary_v2.md; cat $RESULTS_DIR/reports/exp4_summary_v2lora.md

## Stage 3 - Exp5 (ablations), figures, tables
Exp5 needs the Exp3 run `v2` (R1 ranking, k/gamma, R2 gates). About 20-30 min on L4 (rough).

In [ ]:
!python scripts/exp5_ablations.py --resume --exp3-run v2 2>&1 | grep -v -E 'Loading weights|UserWarning|warnings.warn|it/s]'
!grep -E 'crashed|Traceback' $RESULTS_DIR/exp5/main/log.txt | tail -3

In [ ]:
!python scripts/make_figures.py --exp3-run v2 --exp5-run main && python scripts/make_tables.py --exp3-run v2 --exp4-run v2lora --exp5-run main
!ls $RESULTS_DIR/reports/figs/fig* $RESULTS_DIR/reports/tables/table?_*.tex; head -60 $RESULTS_DIR/reports/exp5_summary_main.md

## Stage 4 - real-world evidence, seeds, other TSFMs
Run in this order (most important first); each is resumable and independent. Rough L4 times: Exp6 1-2 h (28 fev-bench covariate tasks + semi-synthetic EPF), Exp8 ~30-60 min (Chronos-2, TimesFM 3.0, TimesFM 2.5 + XReg), Exp7 ~1 h (R2 x 3 training seeds, test x 3 sample seeds). A100 is much faster.

In [ ]:
!python scripts/exp6_real_world.py --resume --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed' $RESULTS_DIR/exp6/main/log.txt | tail -5; head -50 $RESULTS_DIR/reports/exp6_summary_main.md

In [ ]:
!python scripts/exp8_other_models.py --quick 2>&1 | grep -E '\| g |FAILED|skipped|Error|Traceback' | tail -8
!python scripts/exp8_other_models.py --resume 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed|skipped' $RESULTS_DIR/exp8/main/log.txt | tail -5; cat $RESULTS_DIR/reports/exp8_summary_main.md

In [ ]:
!python scripts/exp7_seeds.py --resume --exp3-run v2 2>&1 | grep -v -E 'Loading weights|UserWarning|warnings.warn|it/s]'
!grep -E 'crashed' $RESULTS_DIR/exp7/main/log.txt | tail -3; head -30 $RESULTS_DIR/reports/exp7_summary_main.md

## Stage 5 - multisine diagnosis, semi-synthetic on all fev tasks, cost
Re-run cell 2 first to pull the new scripts. Rough A100 times: Exp9 ~20-30 min (190 cells), Exp10 ~30-60 min (28 tasks x 3 systems x 2 effect sizes, incl. data download and the fine-tuned models), Exp11 ~5 min.

In [ ]:
!python scripts/exp9_multisine.py --quick --resume 2>&1 | grep -E 'cells to run|summary|Error|Traceback'
!python scripts/exp9_multisine.py --resume --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed' $RESULTS_DIR/exp9/main/log.txt | tail -5; cat $RESULTS_DIR/reports/exp9_summary_main.md

In [ ]:
!python scripts/exp10_semisynth_fev.py --resume --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]|symlinks'
!grep -E 'FAILED|crashed|skipped' $RESULTS_DIR/exp10/main/log.txt | tail -8; head -60 $RESULTS_DIR/reports/exp10_summary_main.md

In [ ]:
!python scripts/exp11_cost.py --exp3-run v2 2>&1 | grep -E 'exp11:|Error|Traceback'
!cat $RESULTS_DIR/reports/exp11_summary_main.md

## Stage 6 - Low Carbon London dToU pricing trial: data and model-free ground truth
No GPU needed (a CPU runtime is fine; cells 1-3 and 5 are enough). Re-run cell 2 first. Downloads the London Datastore files (CC BY 4.0, ~800 MB zip; verified by scripts/lcl_download.py) to local disk, builds the household x half-hour matrix (~1 GB, copied to Drive under `pakdd27/data/lcl/processed`), then estimates the price effects with the Std group as control (`$RESULTS_DIR/lcl/feasibility`). Rough time: download 5-15 min, prepare 15-30 min, feasibility < 5 min.

In [ ]:
# verifies size / zip structure / md5; a bad file is reported (first bytes) and replaced; falls back to
# LCL-FullData.zip if the partitioned zip cannot be fetched. The log is kept on Drive.
!mkdir -p /content/lcl /content/drive/MyDrive/pakdd27/data/lcl
!python scripts/lcl_download.py --raw /content/lcl/raw 2>&1 | tee /content/lcl/download.log; cp /content/lcl/download.log /content/drive/MyDrive/pakdd27/data/lcl/
!ls -l /content/lcl/raw

In [ ]:
# prints the zip layout, the first file's columns and a time-parse check before processing
!rm -f /content/drive/MyDrive/pakdd27/data/lcl/processed/hh_kwh.tmp.npy
!python scripts/lcl_prepare.py --raw /content/lcl/raw --out /content/lcl/processed > /content/lcl/prepare.log 2>&1; head -20 /content/lcl/prepare.log; echo ...; tail -40 /content/lcl/prepare.log
!mkdir -p /content/drive/MyDrive/pakdd27/data/lcl && cp /content/lcl/prepare.log /content/drive/MyDrive/pakdd27/data/lcl/
# copy the data to Drive only if preparation finished
!test -s /content/lcl/processed/prepare_log.json && cp -r /content/lcl/processed /content/drive/MyDrive/pakdd27/data/lcl/ && ls -l /content/drive/MyDrive/pakdd27/data/lcl/processed || echo 'PREPARE FAILED - nothing copied'

In [ ]:
!python scripts/lcl_feasibility.py --data /content/lcl/processed > /content/lcl/feasibility.log 2>&1; tail -60 /content/lcl/feasibility.log
!mkdir -p $RESULTS_DIR/lcl/feasibility && cp /content/lcl/feasibility.log $RESULTS_DIR/lcl/feasibility/ && ls -l $RESULTS_DIR/lcl/feasibility

## Stage 7 - Exp12: Chronos-2 what-if on the LCL pricing trial
GPU runtime. Needs only the model-free truth runs already on Drive (`$RESULTS_DIR/lcl/feasibility*`, built locally from the Kaggle mirror) and the Exp3 artefacts. Resumable; `--ft` must stay (it is part of the run's config). ~5-10 min on L4 / A100.

In [ ]:
!python scripts/exp12_lcl_whatif.py --resume --ft 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn'
!grep -E 'FAILED|crashed' $RESULTS_DIR/exp12/main/log.txt | tail -5; cat $RESULTS_DIR/reports/exp12_summary_main.md

## Stage 8 - Exp14: training-free gate (protocol docs/gate_protocol.md)
GPU runtime (A100 or L4). Run Stage 7 first: the LCL check reads all four Exp12 cells. Inputs on Drive: `pakdd27/data/sysid/processed` (Exp13 rigs) and `pakdd27/data/daisy/processed` (DaISy, prepared by scripts/daisy_prepare.py), Exp3/6/12/13 runs. Order matters: select (thresholds are frozen in `exp14/main/gate_select.json`) -> confirm on DaISy -> descriptive checks. Every cell is resumable. Rough time on A100: quick 3-5 min, select 20-40 min (EPF download + 160 semi cells), confirm ~5 min, describe 40-70 min (fev data download dominates).

In [ ]:
# 8a quick check of the whole pipeline (separate run 'quick'; its thresholds are not used anywhere)
!python scripts/exp14_gate.py --quick --resume --part select confirm describe --sysid-data /content/drive/MyDrive/pakdd27/data/sysid/processed --daisy-data /content/drive/MyDrive/pakdd27/data/daisy/processed 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]|symlinks'
!grep -E 'FAILED|crashed|Traceback' $RESULTS_DIR/exp14/quick/log.txt | tail -5; head -30 $RESULTS_DIR/reports/exp14_summary_quick.md

In [ ]:
# 8b selection on simulated / semi-synthetic data -> thresholds frozen
!python scripts/exp14_gate.py --resume --part select --sysid-data /content/drive/MyDrive/pakdd27/data/sysid/processed --daisy-data /content/drive/MyDrive/pakdd27/data/daisy/processed 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]|symlinks'
!grep -E 'FAILED|crashed|Traceback|incomplete' $RESULTS_DIR/exp14/main/log.txt | tail -5; head -20 $RESULTS_DIR/reports/exp14_summary_main.md

In [ ]:
# 8c confirmation on DaISy (pre-declared C1-C3)
!python scripts/exp14_gate.py --resume --part confirm --sysid-data /content/drive/MyDrive/pakdd27/data/sysid/processed --daisy-data /content/drive/MyDrive/pakdd27/data/daisy/processed 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]|symlinks'
!grep -E 'FAILED|crashed|Traceback' $RESULTS_DIR/exp14/main/log.txt | tail -5; sed -n '/Confirmation/,/Descriptive/p' $RESULTS_DIR/reports/exp14_summary_main.md

In [ ]:
# 8d descriptive checks on data seen before the protocol (Exp13, Exp3 test, LCL, fev-bench)
!python scripts/exp14_gate.py --resume --part describe --sysid-data /content/drive/MyDrive/pakdd27/data/sysid/processed --daisy-data /content/drive/MyDrive/pakdd27/data/daisy/processed 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]|symlinks'
!grep -E 'FAILED|crashed|Traceback' $RESULTS_DIR/exp14/main/log.txt | tail -5; sed -n '/Descriptive/,$p' $RESULTS_DIR/reports/exp14_summary_main.md